# Binary-Feedback Template Reconstruction

## 1. Experiment setup

Expected directory structure:

```text
data/
└── lfw/
    └── facenet/
        ├── train_validation/
        │   └── train_validation.npy
        └── test_targets/
            └── test_targets.npy
```

The default setup matches the full experiment:

- 300 target templates
- up to 9,000 breaking-set probes
- 20 binary-search steps per boundary point
- \(d+1\) boundary points for a \(d\)-dimensional SED template

In [1]:
from pathlib import Path
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 6)

# Fixed configuration: LFW / FaceNet / SED / FMR 1%
DATA_ROOT = Path("data")

BREAKING_SET_PATH = (
    DATA_ROOT / "lfw" / "facenet" /
    "train_validation" / "train_validation.npy"
)

TARGETS_PATH = (
    DATA_ROOT / "lfw" / "facenet" /
    "test_targets" / "test_targets.npy"
)

THRESHOLD = 1.1013902272610598
FMR_LABEL = "FMR 1%"

N_TARGETS = 300
N_BREAKING_SET = 9000
BINARY_STEPS = 20

# Seed used by this configuration in the full calibrated experiment.
SEED = 1

print("Configuration")
print("-------------")
print("Dataset:         LFW")
print("Extractor:       FaceNet")
print("Geometry:        Squared Euclidean Distance")
print(f"Operating point: {FMR_LABEL}")
print(f"Threshold:       {THRESHOLD:.16f}")
print(f"Targets:         {N_TARGETS}")
print(f"Breaking set:    {N_BREAKING_SET}")
print(f"Binary steps:    {BINARY_STEPS}")

Configuration
-------------
Dataset:         LFW
Extractor:       FaceNet
Geometry:        Squared Euclidean Distance
Operating point: FMR 1%
Threshold:       1.1013902272610598
Targets:         300
Breaking set:    9000
Binary steps:    20


## 2. Load the embeddings

In [2]:
if not BREAKING_SET_PATH.exists():
    raise FileNotFoundError(
        f"Missing breaking-set embeddings: {BREAKING_SET_PATH}"
    )

if not TARGETS_PATH.exists():
    raise FileNotFoundError(
        f"Missing target embeddings: {TARGETS_PATH}"
    )

breaking_set = np.asarray(np.load(BREAKING_SET_PATH), dtype=np.float64)
targets = np.asarray(np.load(TARGETS_PATH), dtype=np.float64)

if breaking_set.ndim != 2 or targets.ndim != 2:
    raise ValueError(
        f"Expected 2-D arrays, got breaking_set={breaking_set.shape}, "
        f"targets={targets.shape}"
    )

if breaking_set.shape[1] != targets.shape[1]:
    raise ValueError(
        f"Feature dimensions differ: breaking_set={breaking_set.shape[1]}, "
        f"targets={targets.shape[1]}"
    )

feature_dim = targets.shape[1]
n_targets = min(N_TARGETS, len(targets))
n_breaking = min(N_BREAKING_SET, len(breaking_set))

print(f"Breaking set: {breaking_set.shape}")
print(f"Targets:      {targets.shape}")
print(f"Feature dim:  {feature_dim}")
print(f"Using:        {n_targets} targets × {n_breaking} breaking-set probes")

Breaking set: (9938, 512)
Targets:      (466, 512)
Feature dim:  512
Using:        300 targets × 9000 breaking-set probes


## 3. Minimal SED matcher and reconstruction algorithm

The attack uses only a binary matcher:

```text
ACCEPT  if squared_euclidean_distance(probe, target) < threshold
REJECT  otherwise
```


In [3]:
EPS = 1e-12


class NoAcceptedStartingPointError(RuntimeError):
    pass


def squared_euclidean_distance(a, b):
    # Squared Euclidean distance used by the matcher.
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.sum((a - b) ** 2))


def authenticate(target, probe, threshold=THRESHOLD):
    # Binary matcher: True = accept, False = reject.
    return squared_euclidean_distance(probe, target) < threshold


def random_step_sed(x, step_size):
    # Random Euclidean step with squared length = step_size.
    direction = np.random.randn(*x.shape)
    norm = np.linalg.norm(direction)

    while norm <= EPS:
        direction = np.random.randn(*x.shape)
        norm = np.linalg.norm(direction)

    direction /= norm
    return np.asarray(x, dtype=np.float64) + direction * math.sqrt(step_size)


def find_inside_point(data, target, threshold, max_items):
    max_items = min(max_items, len(data))

    for i in range(max_items):
        probe = np.asarray(data[i], dtype=np.float64)
        if authenticate(target, probe, threshold):
            return probe.copy(), i + 1



def find_outside_point(point_inside, target, threshold):
    # Keep trying until the random step is genuinely outside.
    queries = 0

    while True:
        candidate = random_step_sed(point_inside, threshold * 2.0)
        queries += 1

        if not authenticate(target, candidate, threshold):
            return candidate, queries


def find_boundary_point(point_inside, target, threshold, binary_steps):
    inside = np.asarray(point_inside, dtype=np.float64).copy()
    outside, outside_queries = find_outside_point(
        inside, target, threshold
    )

    middle = inside.copy()

    for _ in range(binary_steps):
        middle = (inside + outside) / 2.0

        if authenticate(target, middle, threshold):
            inside = middle
        else:
            outside = middle

    return middle, outside_queries + binary_steps, outside_queries


def solve_sphere_center(points):
    # Equal-radius sphere equations become linear after subtracting one equation.
    q = np.asarray(points, dtype=np.float64)

    if q.ndim != 2 or len(q) < 2:
        raise ValueError("Need a 2-D array with at least two boundary points.")

    q_last = q[-1]
    A = 2.0 * (q[:-1] - q_last)
    b = (
        np.sum(q[:-1] ** 2, axis=1)
        - np.sum(q_last ** 2)
    )

    recovered, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return recovered


def binary_reconstruct(
    data,
    target,
    threshold=THRESHOLD,
    binary_steps=BINARY_STEPS,
    max_breaking=N_BREAKING_SET,
):
    point_inside, inside_queries = find_inside_point(
        data, target, threshold, max_breaking
    )

    d = target.shape[0]
    boundary_points = []
    boundary_queries = 0
    outside_queries = 0

    # d+1 boundary points for a d-dimensional SED sphere.
    for _ in range(d + 1):
        point, q, q_outside = find_boundary_point(
            point_inside,
            target,
            threshold,
            binary_steps,
        )
        boundary_points.append(point)
        boundary_queries += q
        outside_queries += q_outside

    recovered = solve_sphere_center(boundary_points)

    return recovered, {
        "queries": inside_queries + boundary_queries,
        "inside_queries": inside_queries,
        "boundary_queries": boundary_queries,
        "outside_search_queries": outside_queries,
    }


def averaging_baseline(
    data,
    target,
    threshold=THRESHOLD,
    max_breaking=N_BREAKING_SET,
):
    max_items = min(max_breaking, len(data))
    accepted = []

    for i in range(max_items):
        probe = np.asarray(data[i], dtype=np.float64)

        if authenticate(target, probe, threshold):
            accepted.append(probe)


    recovered = np.mean(np.stack(accepted, axis=0), axis=0)

    return recovered, {
        "queries": max_items,
        "accepted_count": len(accepted),
    }

## 4. Run the binary reconstruction

In [4]:
np.random.seed(SEED)

binary_rows = []

for index, target in enumerate(
    tqdm(targets[:n_targets], total=n_targets, desc="Binary reconstruction")
):
    start = time.time()
    recovered, query_info = binary_reconstruct(
        breaking_set,
        target,
        threshold=THRESHOLD,
        binary_steps=BINARY_STEPS,
        max_breaking=n_breaking,
    )

    loss = squared_euclidean_distance(recovered, target)

    binary_rows.append({
        "target": index,
        "queries": query_info["queries"],
        "inside_queries": query_info["inside_queries"],
        "boundary_queries": query_info["boundary_queries"],
        "outside_search_queries": query_info["outside_search_queries"],
        "reconstruction_loss": loss,
        "loss_below_1pct_threshold": loss < 0.01 * THRESHOLD,
        "runtime_sec": time.time() - start,
    })

binary_df = pd.DataFrame(binary_rows)

## 5. Run the averaging baseline

In [5]:
baseline_rows = []

for index, target in enumerate(
    tqdm(targets[:n_targets], total=n_targets, desc="Averaging baseline")
):
    start = time.time()

    recovered, info = averaging_baseline(
        breaking_set,
        target,
        threshold=THRESHOLD,
        max_breaking=n_breaking,
    )

    loss = squared_euclidean_distance(recovered, target)

    baseline_rows.append({
        "target": index,
        "queries": info["queries"],
        "accepted_count": info["accepted_count"],
        "reconstruction_loss": loss,
        "loss_below_1pct_threshold": loss < 0.01 * THRESHOLD,
        "runtime_sec": time.time() - start,
    })


baseline_df = pd.DataFrame(baseline_rows)

## 6. Results

Skipped targets are reported explicitly and excluded from query/loss statistics.

In [6]:
def summarize(df, method):
    n_total = len(df)

    return {
        "Method": method,
        "Targets": n_total,
        "ASR (%)": (
            100.0 * df["loss_below_1pct_threshold"].mean()
            if not df.empty
            else np.nan
        ),
        "Avg. queries": df["queries"].mean(),
        "Query SD": df["queries"].std(ddof=1),
        "Avg. reconstruction loss": df["reconstruction_loss"].mean(),
        "Loss SD": df["reconstruction_loss"].std(ddof=1),
    }


summary = pd.DataFrame([
    summarize(binary_df, "Binary reconstruction"),
    summarize(baseline_df, "Averaging baseline"),
])

display_summary = summary.copy()

for col in ["ASR (%)"]:
    display_summary[col] = display_summary[col].map(lambda x: f"{x:.2f}")

for col in ["Avg. queries", "Query SD"]:
    display_summary[col] = display_summary[col].map(lambda x: f"{x:,.1f}")

for col in ["Avg. reconstruction loss", "Loss SD"]:
    display_summary[col] = display_summary[col].map(lambda x: f"{x:.3e}")

display(
    display_summary.style
    .hide(axis="index")
    .set_caption(
        "LFW / FaceNet / SED / FMR 1% — Binary Reconstruction vs. Averaging Baseline"
    )
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 12px",
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "600"),
                ("text-align", "left"),
                ("padding", "0 0 12px 0"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-weight", "600"),
                ("padding", "8px 12px"),
            ],
        },
    ])
)

Method,Targets,ASR (%),Avg. queries,Query SD,Avg. reconstruction loss,Loss SD
Binary reconstruction,300,99.33,"11,047.5",545.7,1.325e-03,1.794e-02
Averaging baseline,300,0.00,"9,000.0",0.0,3.443e-01,4.771e-02
